<a href="https://colab.research.google.com/github/05haileywang/ECON3916_PS2/blob/main/ps2_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [58]:
# Step 1.1

def population(n, base, period):
    # Calculate engagement minutes; lognormal because engagement time is not normally distributed
    engagement = rng.lognormal(mean = 3.5, sigma = 0.6, size = n)
    # Calculate z-score for engagement minutes
    engagement_z = (engagement - engagement.mean()) / engagement.std()
    # Bernoulli distribution (0 = no support ticket; 1 = support ticket)
    had_support_ticket = rng.binomial(1, 0.22, n)

    s = base + 0.35 * engagement_z - 0.60 * had_support_ticket + rng.normal(0, 0.75, size=n)

    s = np.round(s)
    s = np.clip(s, 1, 5)
    s = s.astype(int)

    return pd.DataFrame({
        "period": period,
        "engagement_minutes": engagement,
        "engagement_z": engagement_z,
        "had_support_ticket": had_support_ticket,
        "satisfaction": s
    })

pre = population(50000, 3.432, "Before")
post = population(50000, 3.532, "After")

true_pre = pre["satisfaction"].mean()
true_post = post["satisfaction"].mean()

print(f"True pre-redesign mean: {true_pre:.2f}")
print(f"True post-redesign mean: {true_post:.2f}")
print(f"True improvement: {true_post - true_pre:.2f}")

True pre-redesign mean: 3.29
True post-redesign mean: 3.39
True improvement: 0.10


*Your answer here.*

In part 1.1, I simulated a population of 50,000 users before and after implementing a redesign. They were randomly assigned an engagement times, submission of a support ticket, and satisfaction score (1-5). Satisfaction was calculated based on the listed parameters. I used a random seed to ensure that the results were reproducible.

The true population mean before the redesign was approximately 3.29, and the mean after the redesign was 3.39. This is an improvement of 0.10 points. Based on these results, we are able to conduct further analysis and determine if the nonresponse bias affects the estimates.

### Step 1.2: Impose the response mechanism (15 pts)

In [59]:
# Step 1.2: Impose the Response Mechanism

# Function to calculate response probabilities
def response(df, a, b):

    z = (a + b * (df["satisfaction"] - 3.3)
         + 0.30 * df["engagement_z"]
         - 0.25 * df["had_support_ticket"])

    # Calculate probability of responding
    p_respond = 1 / (1 + np.exp(-z))

    df = df.copy()
    df["p_respond"] = p_respond

    # Determine whether each user responds
    df["responded"] = (rng.random(len(df)) < p_respond).astype(int)

    return df

# Apply response mechanism before and after redesign
pre = response(pre, a=-1.0, b=0.45)
post = response(post, a=-2.75, b=2.00)

# Select users who responded
pre_respondents = pre[pre["responded"] == 1]
post_respondents = post[post["responded"] == 1]

# Calculate observed satisfaction means
pre_observed_mean = pre_respondents["satisfaction"].mean()
post_observed_mean = post_respondents["satisfaction"].mean()

# Calculate observed improvement
observed_difference = post_observed_mean - pre_observed_mean

# Calculate response rates
pre_response_rate = pre["responded"].mean()
post_response_rate = post["responded"].mean()

# Results
print(f"Observed pre-redesign mean: {pre_observed_mean:.4f}")
print(f"Observed post-redesign mean: {post_observed_mean:.4f}")
print(f"Observed improvement: {observed_difference:.4f}")

print(f"\nPre-redesign response rate: {pre_response_rate:.2%}")
print(f"Post-redesign response rate: {post_response_rate:.2%}")

print(f"\nTrue population improvement: {post['satisfaction'].mean() - pre['satisfaction'].mean():.4f}")

Observed pre-redesign mean: 3.6179
Observed post-redesign mean: 4.3594
Observed improvement: 0.7415

Pre-redesign response rate: 27.50%
Post-redesign response rate: 15.57%

True population improvement: 0.0959


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

*Your answer here.*

This is an example of nonresponse bias because people who are more satisfied with the company's product, both pre- and post-redesign (higher satisfaction scores), are more likely to respond to a survey. The observed satisfaction scores would be higher than the true population means due to nonresponse bias. A larger survey would not fix this type of bias because it would still disproportionately represent users who are more satisfied with the product. The pre-redesign response rate was 27.50%, while the post-redesign response rate was 15.57%, which reflects the fact that the survey that produced the better outcome heard from fewer people. The observed improvement was 0.7415 points; however, the true population improvement was only 0.0959 points, which demonstrates how nonresponse bias can overstate the apparent impact of the redesign.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [60]:
# Step 2.1: Bootstrap by Hand

scores = post_respondents["satisfaction"].to_numpy()

B = 9999

# Create empty lists
bootstrap_means = []
bootstrap_medians = []

for i in range(B):
    sample = rng.choice(scores, size=len(scores), replace=True)

    # Appends the bootstrapped means to the empty lists
    bootstrap_means.append(np.mean(sample))
    bootstrap_medians.append(np.median(sample))

mean_ci = np.percentile(bootstrap_means, [2.5, 97.5])
median_ci = np.percentile(bootstrap_medians, [2.5, 97.5])

print(f"Observed mean: {np.mean(scores):.2f}")
print(f"Mean 95% percentile CI: {mean_ci}")

print(f"Observed median: {np.median(scores):.2f}")
print(f"Median 95% percentile CI: {median_ci}")

Observed mean: 4.36
Mean 95% percentile CI: [4.34515093 4.3734104 ]
Observed median: 4.00
Median 95% percentile CI: [4. 4.]


In [61]:
# Step 2.1: Bootstrap using scipy.stats.bootstrap

from scipy.stats import bootstrap

# Mean percentile interval
scipy_mean_percentile = bootstrap(
    (scores,),
    np.mean,
    confidence_level=0.95,
    n_resamples=9999,
    method="percentile",
    random_state=1000
)

# Mean BCa interval
scipy_mean_bca = bootstrap(
    (scores,),
    np.mean,
    confidence_level=0.95,
    n_resamples=9999,
    method="BCa",
    random_state=1000
)

# Median percentile interval
scipy_median_percentile = bootstrap(
    (scores,),
    np.median,
    confidence_level=0.95,
    n_resamples=9999,
    method="percentile",
    random_state=1000
)

# Median BCa interval
scipy_median_bca = bootstrap(
    (scores,),
    np.median,
    confidence_level=0.95,
    n_resamples=9999,
    method="BCa",
    random_state=1000
)

# Outputs mean confidence intervals
print("95% Confidence Intervals for Mean")
print(f"Manual Percentile: [{mean_ci[0]:.3f}, {mean_ci[1]:.3f}]")
print(f"SciPy Percentile:  [{scipy_mean_percentile.confidence_interval.low:.3f}, {scipy_mean_percentile.confidence_interval.high:.3f}]")
print(f"SciPy BCa:         [{scipy_mean_bca.confidence_interval.low:.3f}, {scipy_mean_bca.confidence_interval.high:.3f}]")

# Outputs median confidence intervals
print("\n95% Confidence Intervals for Median")
print(f"Manual Percentile: [{median_ci[0]:.3f}, {median_ci[1]:.3f}]")
print(f"SciPy Percentile:  [{scipy_median_percentile.confidence_interval.low:.3f}, {scipy_median_percentile.confidence_interval.high:.3f}]")
print(f"SciPy BCa:         [{scipy_median_bca.confidence_interval.low:.3f}, {scipy_median_bca.confidence_interval.high:.3f}]")

95% Confidence Intervals for Mean
Manual Percentile: [4.345, 4.373]
SciPy Percentile:  [4.345, 4.374]
SciPy BCa:         [4.345, 4.374]

95% Confidence Intervals for Median
Manual Percentile: [4.000, 4.000]
SciPy Percentile:  [4.000, 4.000]
SciPy BCa:         [nan, nan]


/usr/local/lib/python3.13/dist-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:352: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


I manually bootstrapped the data by generating 9,999 bootstrap samples by randomly selecting with replacement. Then I calculated the mean and median for each sample. I then used the bootstrap function from SciPy to generate and verify my results.

The manual and SciPy percentile confidence intervals are nearly the same, which suggests that my manual bootstrap was implemented correctly. The SciPy BCa interval for the mean is close to the percentile interval, demonstrating that the adjustment for bootstrap bias and acceleration did not make a substantial difference.

For the median, both the manual and SciPy methods output a confidence interval of [4.000, 4.000], while the BCa method output an undefined interval. This is likely because the satisfaction scale is from 1 to 5, and many of the bootstrap samples end up having the same median of 4. As a result, there is not enough variation for the BCa method to calculate its adjustment.

These confidence intervals are not designed to correct nonresponse bias, so they may not contain the true population mean or median.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [62]:
# Step 2.2: Coverage Simulation

# Number of repeated surveys and bootstrap samples
R = 200
B = 500

# True population mean and median
true_mean = post["satisfaction"].mean()
true_median = post["satisfaction"].median()

# Creates empty lists to store coverage results
mean_covered = []
median_covered = []

# Repeats the survey 200 times
for i in range(R):

    # Simulates new survey responses using the original response probabilities
    responded = (rng.random(len(post)) < post["p_respond"]).astype(int)

    # Selects users who responded
    survey = post.loc[responded == 1, "satisfaction"].to_numpy()

    # Creates empty lists for bootstrap statistics
    boot_means = []
    boot_medians = []

    # Generates 500 bootstrap samples
    for j in range(B):
        sample = rng.choice(survey, size=len(survey), replace=True)

        boot_means.append(np.mean(sample))
        boot_medians.append(np.median(sample))

    # Calculates 95% percentile confidence intervals
    mean_ci = np.percentile(boot_means, [2.5, 97.5])
    median_ci = np.percentile(boot_medians, [2.5, 97.5])

    # Checks whether confidence intervals contain the true values
    mean_covered.append(mean_ci[0] <= true_mean <= mean_ci[1])
    median_covered.append(median_ci[0] <= true_median <= median_ci[1])

# Calculates coverage rates
mean_coverage = np.mean(mean_covered)
median_coverage = np.mean(median_covered)

# Calculates Monte Carlo standard errors
mean_mc_se = np.sqrt(mean_coverage * (1 - mean_coverage) / R)
median_mc_se = np.sqrt(median_coverage * (1 - median_coverage) / R)

# Results
print(f"Mean coverage: {mean_coverage:.2%}")
print(f"Mean Monte Carlo SE: {mean_mc_se:.2%}")

print(f"\nMedian coverage: {median_coverage:.2%}")
print(f"Median Monte Carlo SE: {median_mc_se:.2%}")

Mean coverage: 0.00%
Mean Monte Carlo SE: 0.00%

Median coverage: 0.00%
Median Monte Carlo SE: 0.00%


*Your answer here.*

I simulated 200 repeated surveys, with 500 bootstrap resamples per survey. The code calculates 95% confidence intervals for the mean and median, and then checks whether or not these intervals contain the true population parameters.

The mean and median had a coverage rate of 0%, along with a Monte Carlo standard error of 0%, meaning that none of the confidence intervals generated in the simulation contained the true population mean or median. The Monte Carlo standard error is 0% because the estimated coverage is zero. However, this does not necessarily mean that there is no uncertainty in the true coverage probability.

Highly satisfied users are more likely to respond to the survey, creating nonresponse bias. As a result, the bootstrap confidence intervals are centered around the biased respondent estimates and do not contain the true population mean or median.

The coverage rate of 0% is substantially below the expected 95% coverage, demonstrating that the confidence intervals are not accurately capturing the true population parameters. Bootstrapping does not correct for nonresponse bias, even when the confidence intervals are narrow.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [63]:
# Step 3.1

from scipy.stats import ttest_ind

pre_scores = pre_respondents["satisfaction"]
post_scores = post_respondents["satisfaction"]

t_stat, p_value = ttest_ind(post_scores, pre_scores, equal_var=False)

difference = post_scores.mean() - pre_scores.mean()

print(f"Mean difference: {difference:.4f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")

# Calculate Cohen's d

n1 = len(post_scores)
n2 = len(pre_scores)

sd1 = post_scores.std()
sd2 = pre_scores.std()

pooled_sd = np.sqrt(((n1 - 1) * sd1**2 + (n2 - 1) * sd2**2) / (n1 + n2 - 2))

cohens_d = difference / pooled_sd

print(f"Cohen's d: {cohens_d:.4f}")

Mean difference: 0.7415
T-statistic: 71.1231
P-value: 0.000000
Cohen's d: 0.9267


*Your answer here.*

Step 3.1: A Welch's two-sample t-test showed that the mean difference was 0.7415, with a t-statistic of 71.1231 and a p-value below 0.001. Cohen's d was 0.9267, which means that there is a large effect size. The p-value is below 0.05, which means that we reject the null hypothesis that the pre- and post-redesign respondent means are equal.

The observed survey results showed a 0.7415-point difference in satisfaction. However, the true population improvement is only approximately 0.10 points. Highly satisfied users are more likely to respond to surveys, especially after the redesign was implemented. We can conclude that the t-test showed a statistically significant difference between the two groups, but this type of test does not correct for nonresponse bias. Therefore, statistical significance does not necessarily mean that the redesign caused an improvement of this size.

### Step 3.2: Power, before the fact

In [64]:
# Step 3.2: Power Analysis

from scipy.stats import norm

# Calculates required sample size for 80% power
sd = pre["satisfaction"].std()
effect = 0.10

n = 2 * (1.96 + 0.84)**2 * sd**2 / effect**2

print(f"Required sample size per group: {np.ceil(n):.0f}")

# Calculates minimum detectable effect using available respondents
n_available = min(len(pre_respondents), len(post_respondents))

mde = (1.96 + 0.84) * sd * np.sqrt(2 / n_available)

print(f"Available sample size per group: {n_available}")
print(f"Minimum detectable effect: {mde:.4f}")

# Simulates whether larger samples reduce nonresponse bias

sample_sizes = [500, 1000, 5000]
R = 100

for size in sample_sizes:

    differences = []

    # Repeats the survey 100 times
    for i in range(R):

        # Randomly selects users from each population
        pre_sample = pre.sample(n=size, replace=True,
                                random_state=int(rng.integers(1000000)))
        post_sample = post.sample(n=size, replace=True,
                                  random_state=int(rng.integers(1000000)))

        # Simulates survey responses
        pre_responses = rng.binomial(1, pre_sample["p_respond"])
        post_responses = rng.binomial(1, post_sample["p_respond"])

        # Calculates mean satisfaction among respondents
        pre_mean = pre_sample.loc[pre_responses == 1, "satisfaction"].mean()
        post_mean = post_sample.loc[post_responses == 1, "satisfaction"].mean()

        # Calculates observed improvement
        differences.append(post_mean - pre_mean)

    # Calculates average improvement and standard deviation
    avg_difference = np.mean(differences)
    sd_difference = np.std(differences, ddof=1)

    print(f"\nSample size: {size}")
    print(f"Average observed improvement: {avg_difference:.4f}")
    print(f"Standard deviation: {sd_difference:.4f}")

Required sample size per group: 1248
Available sample size per group: 7785
Minimum detectable effect: 0.0400

Sample size: 500
Average observed improvement: 0.7330
Standard deviation: 0.1047

Sample size: 1000
Average observed improvement: 0.7276
Standard deviation: 0.0694

Sample size: 5000
Average observed improvement: 0.7420
Standard deviation: 0.0322



*Your answer here.*

1,248 respondents per group would be needed to detect a true improvement of 0.10 points with 80% statistical power and a 5% significance level. The available sample size per group is 7,785 respondents, and the MDE was 0.0400 points.

I simulated 100 surveys for each sample size of 500, 1,000, and 5,000 users. The average observed improvement was 0.7330, 0.7276, and 0.7420 points, respectively. The standard deviation decreased as the sample size increased, from 0.1047 to 0.0694 and then to 0.0322.

A larger sample can help reduce variability, but it is not a fix for nonresponse bias. The observed improvement remained higher than the true population improvement of approximately 0.10 points, even as the sample size increased. Once again, nonresponse bias plays a part in this. The more satisfied a user is, the more likely they are to respond to a survey. Increasing the sample size does not address this nonresponse bias.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [65]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Step 1: Create response indicator
post = post.copy()
post["responded"] = post.index.isin(post_respondents.index).astype(int)

# Step 2: Fit logistic regression
X = post[["engagement_minutes", "had_support_ticket"]]
y = post["responded"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# Step 3: Predict response probabilities
post["response_prob"] = model.predict_proba(X)[:, 1]

# Step 4: Calculate inverse probability weights
respondents = post[post["responded"] == 1].copy()
respondents["weight"] = 1 / respondents["response_prob"]

# Step 5: Calculate means
unweighted_mean = respondents["satisfaction"].mean()

weighted_mean = np.average(
    respondents["satisfaction"],
    weights=respondents["weight"]
)

true_mean = post["satisfaction"].mean()

# Step 6: Bootstrap with model refitting
B = 2000
rng_ipw = np.random.default_rng(3916)

bootstrap_means = []

for i in range(B):

    # Resample the full population
    sample = post.iloc[
        rng_ipw.choice(len(post), size=len(post), replace=True)
    ].copy()

    # Refit logistic regression
    X_sample = sample[["engagement_minutes", "had_support_ticket"]]
    y_sample = sample["responded"]

    model_boot = LogisticRegression(max_iter=1000)
    model_boot.fit(X_sample, y_sample)

    # Recalculate response probabilities
    sample["response_prob"] = model_boot.predict_proba(X_sample)[:, 1]

    # Select respondents
    sample_respondents = sample[sample["responded"] == 1].copy()

    # Recalculate weights
    sample_respondents["weight"] = 1 / sample_respondents["response_prob"]

    # Calculate weighted mean
    mean = np.average(
        sample_respondents["satisfaction"],
        weights=sample_respondents["weight"]
    )

    bootstrap_means.append(mean)

# Step 7: Calculate 95% bootstrap confidence interval
ci = np.percentile(bootstrap_means, [2.5, 97.5])

# Step 8: Outputs
print(f"Unweighted respondent mean: {unweighted_mean:.4f}")
print(f"IPW-adjusted mean: {weighted_mean:.4f}")
print(f"True population mean: {true_mean:.4f}")

print(f"\n95% Bootstrap CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

print(f"\nUnweighted bias: {unweighted_mean - true_mean:.4f}")
print(f"IPW bias: {weighted_mean - true_mean:.4f}")

print(f"\nCI contains true mean: {ci[0] <= true_mean <= ci[1]}")

Unweighted respondent mean: 4.3594
IPW-adjusted mean: 4.2161
True population mean: 3.3860

95% Bootstrap CI: [4.1962, 4.2346]

Unweighted bias: 0.9734
IPW bias: 0.8301

CI contains true mean: False


*Your answer here.*

The true population mean was 3.3860, the unweighted respondent mean was 4.3594, and the IPW-adjusted mean was 4.2161. The IPW correction reduced the bias from 0.9734 to 0.8301. However, the adjusted mean was still greater than the population mean. The 95% bootstrap confidence interval was [4.1962, 4.2346], which does not contain the true population mean.

The weights are calculated by dividing 1 by each respondent's estimated probability of responding. This gives proportionally more weight to users who are less likely to respond. These weights are then used to calculate a new adjusted mean to account for nonresponse bias.

IPW assumes that the probability of responding is independent of satisfaction after controlling for engagement minutes and support ticket history. However, in this case more satisfied users are more likely to respond even after accounting for these variables. This violates the Missing at Random assumption. IPW cannot fully correct the nonresponse bias.

### Task 4.2: The slide (5 pts)

The survey results showed that satisfaction increased after the onboarding redesign. However, highly satisfied users are more likely to respond to surveys, which creates nonresponse bias. This is especially true after the redesign was implemented. As a result, the observed improvement may not accurately reflect the true population improvement.

The post-redesign respondent mean was 4.3594, while the IPW-adjusted mean was 4.2161, with a 95% bootstrap confidence interval of [4.1962, 4.2346]. However, the true population improvement was only approximately 0.10 points. Although IPW reduced the nonresponse bias, the adjusted mean was still higher than the true population mean. We can conclude that the survey results showed an improvement in satisfaction, but they do not necessarily demonstrate that the redesign caused an improvement of this size.

To get more accurate results, the company should randomly select users to participate in the survey and follow up with users who do not respond. The company should also use the same survey process before and after the redesign to reduce nonresponse bias and make the results more comparable.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

**Model and date:** ChatGPT Instant GPT-6 on 10/26/26

**Version 1 prompt:**

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

**Version 1 reply:**

# Inverse Probability Weighting (IPW)

Inverse Probability Weighting attempts to correct nonresponse bias by assigning greater weight to respondents who were less likely to complete the survey.

Since you already have `post`, `post_respondents`, `engagement_minutes`, `had_support_ticket`, and `satisfaction`, we can reuse those variables.

### Step 1: Estimate Response Probabilities

We'll use logistic regression to predict whether someone responds based on their engagement and support ticket history.

```python
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# Create response indicator
post["responded"] = post.index.isin(post_respondents.index).astype(int)

# Covariates
X = post[["engagement_minutes", "had_support_ticket"]]
y = post["responded"]

# Fit logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# Predict response probabilities
post["response_prob"] = model.predict_proba(X)[:, 1]
```

This assumes `post` contains the full simulated population and that `post_respondents` retains the original row indices from `post`.

### Step 2: Calculate the Weighted Mean

Each respondent receives a weight equal to the inverse of their estimated response probability.

$$
w_i = \frac{1}{\hat{p}_i}
$$

```python
# Select respondents
respondents = post[post["responded"] == 1].copy()

# Calculate weights
respondents["weight"] = 1 / respondents["response_prob"]

# Calculate weighted mean
weighted_mean = np.average(
    respondents["satisfaction"],
    weights=respondents["weight"]
)

print(f"Unweighted mean: {respondents['satisfaction'].mean():.4f}")
print(f"IPW mean: {weighted_mean:.4f}")
print(f"True population mean: {post['satisfaction'].mean():.4f}")
```

This uses a normalized (Hájek) IPW estimator.

### Step 3: Bootstrap Confidence Interval

To estimate uncertainty, we can resample respondents and calculate the weighted mean repeatedly.

```python
B = 2000
bootstrap_means = []

for i in range(B):
    sample = respondents.sample(
        n=len(respondents),
        replace=True
    )

    mean = np.average(
        sample["satisfaction"],
        weights=sample["weight"]
    )

    bootstrap_means.append(mean)

ci = np.percentile(bootstrap_means, [2.5, 97.5])

print(f"IPW mean: {weighted_mean:.4f}")
print(f"95% bootstrap CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
```

This is a basic bootstrap treating estimated weights as fixed. A more complete bootstrap would resample the full population records and refit the logistic regression in each iteration to account for uncertainty in the response model.

### Mechanism Check

**What assumption makes IPW valid?**

IPW requires that the probability of responding is independent of satisfaction after controlling for the observed covariates.

This is called the Missing at Random (MAR) assumption:

$$
R \perp Y \mid X
$$

Where:

- $R$ = whether someone responds
- $Y$ = satisfaction
- $X$ = engagement minutes and support ticket history

In other words, once we account for engagement and support tickets, satisfaction itself must not influence whether someone responds.

We also need positive response probabilities for all relevant covariate groups and a correctly specified response probability model.

**What happens when response depends on satisfaction itself?**

Your simulation makes more satisfied users more likely to respond, even after accounting for engagement and support tickets.

This violates MAR because response depends directly on satisfaction, which is unobserved for nonrespondents.

This is called Missing Not at Random (MNAR).

### Evaluation: Will IPW Recover the True Mean?

**No, not reliably.**

Because your response mechanism directly depends on satisfaction, the logistic regression cannot fully explain response probabilities using only engagement minutes and support tickets.

Even after weighting, respondents with higher satisfaction remain overrepresented relative to respondents with lower satisfaction who have similar covariates.

As a result, the weighted mean will generally remain biased upward compared with the true population mean.

The correction may reduce some bias, but it cannot eliminate the bias caused by satisfaction-dependent nonresponse.

**Version 2 prompt:**

[Prep] Act as a survey statistician who works on non-response adjustment for product research.

[Request] I have a simulated user population with satisfaction scores and two covariates, engagement_minutes and had_support_ticket, plus a non-response mechanism where the probability of responding increases with satisfaction. Write Python that implements inverse-probability weighting to correct the observed post-redesign survey mean, using a logistic regression model of response on the covariates.

[Iterate] Use numpy, pandas, and scikit-learn or statsmodels. Reuse my existing variable names: post, post_respondents, engagement_minutes, had_support_ticket, and satisfaction. Return the unweighted respondent mean, IPW-adjusted mean, true post-redesign population mean, and a 95% bootstrap confidence interval.

[Bootstrap Correction] In Version 1, the bootstrap treated the estimated inverse-probability weights as fixed. For Version 2, resample the full simulated population with replacement and refit the logistic regression model during each bootstrap iteration. Recalculate the response probabilities and inverse-probability weights for each sample before computing the weighted mean. Use 2,000 bootstrap iterations and a fixed random seed for reproducibility.

[Mechanism Check] Explain precisely which assumption makes this correction valid, and what happens when response depends on satisfaction itself rather than only on the covariates.

[Evaluate] Given my simulated nonresponse mechanism, will this correction recover the true population mean? Answer yes or no and justify it.

**Version 2 reply:**

### Version 2: Inverse Probability Weighting (IPW)

This revised implementation uses logistic regression to estimate response probabilities and inverse probability weighting to adjust the observed post-redesign satisfaction mean.

Unlike Version 1, the bootstrap resamples the full simulated population and refits the logistic regression during every iteration. This accounts for uncertainty in the estimated response probabilities and weights.

### Python Code

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Step 1: Create response indicator
post = post.copy()
post["responded"] = post.index.isin(post_respondents.index).astype(int)

# Step 2: Fit logistic regression
X = post[["engagement_minutes", "had_support_ticket"]]
y = post["responded"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# Step 3: Predict response probabilities
post["response_prob"] = model.predict_proba(X)[:, 1]

# Step 4: Calculate inverse probability weights
respondents = post[post["responded"] == 1].copy()
respondents["weight"] = 1 / respondents["response_prob"]

# Step 5: Calculate means
unweighted_mean = respondents["satisfaction"].mean()

weighted_mean = np.average(
    respondents["satisfaction"],
    weights=respondents["weight"]
)

true_mean = post["satisfaction"].mean()

# Step 6: Bootstrap with model refitting
B = 2000
rng_ipw = np.random.default_rng(3916)

bootstrap_means = []

for i in range(B):

    # Resample the full population
    sample = post.iloc[
        rng_ipw.choice(len(post), size=len(post), replace=True)
    ].copy()

    # Refit logistic regression
    X_sample = sample[["engagement_minutes", "had_support_ticket"]]
    y_sample = sample["responded"]

    model_boot = LogisticRegression(max_iter=1000)
    model_boot.fit(X_sample, y_sample)

    # Recalculate response probabilities
    sample["response_prob"] = model_boot.predict_proba(X_sample)[:, 1]

    # Select respondents
    sample_respondents = sample[sample["responded"] == 1].copy()

    # Recalculate weights
    sample_respondents["weight"] = 1 / sample_respondents["response_prob"]

    # Calculate weighted mean
    mean = np.average(
        sample_respondents["satisfaction"],
        weights=sample_respondents["weight"]
    )

    bootstrap_means.append(mean)

# Step 7: Calculate 95% bootstrap confidence interval
ci = np.percentile(bootstrap_means, [2.5, 97.5])

# Step 8: Outputs
print(f"Unweighted respondent mean: {unweighted_mean:.4f}")
print(f"IPW-adjusted mean: {weighted_mean:.4f}")
print(f"True population mean: {true_mean:.4f}")

print(f"\n95% Bootstrap CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

print(f"\nUnweighted bias: {unweighted_mean - true_mean:.4f}")
print(f"IPW bias: {weighted_mean - true_mean:.4f}")

print(f"\nCI contains true mean: {ci[0] <= true_mean <= ci[1]}")
```

### Mechanism Check

IPW relies on the Missing at Random (MAR) assumption:

\[
R \perp Y \mid X
\]

This means that after controlling for engagement minutes and support ticket history, whether someone responds must be independent of their satisfaction score.

The method also requires positive response probabilities and a correctly specified response model.

However, your simulation makes users with higher satisfaction scores more likely to respond, even after accounting for these covariates. This violates MAR because response probability depends directly on satisfaction itself. This is known as Missing Not at Random (MNAR).

### Evaluation

**No, IPW will not reliably recover the true population mean.**

Although it adjusts for differences in engagement minutes and support ticket history, it cannot fully correct for the direct relationship between satisfaction and response probability.

The revised bootstrap accounts for uncertainty in the estimated response model by refitting it during each iteration. However, this does not eliminate the underlying nonresponse bias.

The resulting confidence interval can therefore be narrow while still failing to contain the true population mean.

**What I Changed and Why:**

**Version 1:** The code calculated the weights once and then used those same weights for every bootstrap sample. This did not account for the fact that the estimated probabilities of responding could change between samples.

**Version 2:** The code takes new samples from the full population and recalculates the probability of responding and the weights each time. This helps account for differences between samples and gives a better estimate of the uncertainty in the results.

### Verification

The true population mean was 3.3860, while the IPW-adjusted mean was 4.2161. Comparing these values shows whether IPW was able to reduce the nonresponse bias in my simulation.

### In My Own Words

The code calculates the weights by dividing 1 by each respondent's estimated probability of responding. This gives higher weights to people who were less likely to respond and lower weights to people who were more likely to respond. The weights are then used to calculate an adjusted satisfaction mean to account for differences in response probabilities.

### Judgment

I would use AI for this step again because it helped me implement IPW, but I would still verify the results and assumptions before relying on them.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.